In [8]:
import numpy as np
import pandas as pd
import ast
import pickle
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC

# --- 1. LOAD DATA ---
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

# --- 2. MERGE ---
movies = movies.merge(credits, on='title')

# --- 3. SELECT COLUMNS (Numeric columns added here to avoid KeyError) ---
movies = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'popularity', 'vote_average', 'vote_count']].copy()

# --- 4. HANDLE NULLS & DUPLICATES ---
movies.dropna(subset=['overview'], inplace=True)
movies.drop_duplicates(subset=['title'], inplace=True)

# Safety check: Fill any empty numeric values with 0 so SVM doesn't fail
movies['popularity'] = movies['popularity'].fillna(0)
movies['vote_count'] = movies['vote_count'].fillna(0)
movies['vote_average'] = movies['vote_average'].fillna(0)

movies.reset_index(drop=True, inplace=True)

# --- 5. DEFINE CLEANING FUNCTIONS ---
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else: break
    return L

def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            # Cleaning space in director name here
            L.append(i['name'].replace(" ",""))
            break
    return L

# --- 6. APPLY CLEANING ---
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# --- 7. REMOVE SPACES ---
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])

# --- 8. CREATE TAGS & NEW DATAFRAME ---
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Keeping all columns needed for the App and the SVM
new_df = movies[['id', 'title', 'tags', 'popularity', 'vote_average', 'vote_count']].copy()

# Process tags
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())
new_df['tags'] = new_df['tags'].apply(stem)

# --- 9. SVM TRAINING ---
# Creating Quality Label: 1 for Good (>=7.0), 0 for Average
new_df['quality_label'] = new_df['vote_average'].apply(lambda x: 1 if x >= 7.0 else 0)

X_svm = new_df[['popularity', 'vote_count']]
y_svm = new_df['quality_label']

svm_model = SVC(kernel='rbf', probability=True)
svm_model.fit(X_svm, y_svm)
print("SVM Model trained successfully!")

# --- 10. VECTORIZATION & SIMILARITY ---
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)

# --- 11. PICKLE EXPORTS ---
pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
pickle.dump(svm_model, open('svm_model.pkl', 'wb'))



print("Compressed similarity file created!")

print(f"Final Movie Count: {len(new_df)}")
new_df.head()

SVM Model trained successfully!
Compressed similarity file created!
Final Movie Count: 4797


,id,title,tags,popularity,vote_average,vote_count,quality_label
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa...",150.437577,7.2,11800,1
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c...",139.082615,6.9,4500,0
2,206647,Spectre,a cryptic messag from bond’ past send him on a...,107.376788,6.3,4466,0
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...,112.312950,7.6,9106,1
4,49529,John Carter,"john carter is a war-weary, former militari ca...",43.926995,6.1,2124,0


In [ ]:
new_df['tags'][0]

In [ ]:
vectors= cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors[0]

In [ ]:
len(cv.get_feature_names_out())

In [ ]:
vectors[0]

In [ ]:
# Convert the features to a list so it doesn't get truncated
feature_names = cv.get_feature_names_out()

# Print the whole list
for word in feature_names:
    print(word).shape

In [ ]:
['loved','loving','love']
['love','love','love']

In [ ]:
ps.stem('dancing')

In [ ]:
 from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
similarity

In [ ]:
sorted(list(enumerate(similarity[0].tolist())),reverse=True,key=lambda x:x[1])[1:6]

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie ].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances.tolist())),reverse=True,key=lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [9]:
import lzma
import pickle

# This takes your 'similarity' matrix and saves it as a compressed .xz file
with lzma.open('similarity.pkl.xz', 'wb') as f:
    pickle.dump(similarity, f)

print("Compression complete! Your file is now 'similarity.pkl.xz'")

Compression complete! Your file is now 'similarity.pkl.xz'
